# Klasifikasi Fakta/Hoaks berbasis Kalimat & Ekstraksi Entitas (NER)

Arsitektur ini berfokus pada efisiensi tinggi menggunakan Google Colab GPU T4.
Terdiri dari dua arsitektur paralel:
1.  **IndoBERT Fine-Tuned (Trainer)**: Mendeteksi teks pada level *Sentence Classification*.
2.  **NER Pipeline Eksternal**: Menangkap entitas nama/lokasi pada kalimat untuk melengkapi konteks deteksi hoaks.
*Catatan: Augmentasi minoritas hanya diterapkan pada set latih (train-only), tidak pada validasi/uji.*


In [1]:
# 1. Instalasi Dependensi Inti
!pip install -q transformers datasets accelerate sentencepiece scikit-learn nltk

In [2]:
# 2. Pengaitan (Mounting) Google Drive
# Wajib dijalankan untuk mendapatkan akses ke folder scrape_output
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 3. Impor Library dan Persiapan Lingkungan GPU
import os
import gc
import shutil
import warnings
import re
import random
import json
import subprocess
import sys
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed,
    pipeline,
)

import nltk
from nltk.tokenize import sent_tokenize
from google.colab import files

warnings.filterwarnings("ignore")
nltk.download("punkt")
nltk.download("punkt_tab")

# Mematikan thread konversi safetensors otomatis di latar belakang
os.environ["HF_HUB_DISABLE_IMPLICIT_SAFETENSORS"] = "1"

print("Library dan Environment berhasil diinisialisasi.")



In [ ]:
# 4. Konfigurasi Arsitektur (Dataset Processed + Training Konsisten Inference)
@dataclass
class Config:
    project_root: str = "/content"  # Sesuaikan jika Anda menjalankan di luar Colab

    # Raw CSV (dipakai oleh scripts/build_dataset.py)
    file_hoaks: str = "data_hoaks_turnbackhoaks.csv"
    file_nonhoaks_cnn: str = "data_nonhoaks_cnn.csv"
    file_nonhoaks_detik: str = "data_nonhoaks_detik.csv"
    file_nonhoaks_kompas: str = "data_nonhoaks_kompas.csv"

    # Processed dataset
    processed_dir: str = "data/processed"
    train_csv: str = "train.csv"
    val_csv: str = "val.csv"
    test_csv: str = "test.csv"

    # Builder
    builder_script: str = "scripts/build_dataset.py"

    # Model
    model_name: str = "indolem/indobert-base-uncased"
    ner_model_name: str = "cahya/bert-base-indonesian-NER"

    # Training setup
    max_length: int = 192
    train_batch_size: int = 16
    eval_batch_size: int = 32
    grad_accumulation: int = 2
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    num_epochs: int = 3
    auto_find_batch_size: bool = True
    early_stopping_patience: int = 2
    early_stopping_threshold: float = 0.0

    # Imbalance
    imbalance_method: str = "class_weight"  # opsi: class_weight | focal
    focal_gamma: float = 2.0

    # Label canonical
    label_fakta: int = 0
    label_hoaks: int = 1

    # Output
    output_dir: str = "indobert_hoax_ner_model_final"

    # Inference/Stress test
    orange_threshold: float = 0.65
    stress_samples_per_source: int = 2

    # NER inferensi
    ner_infer_device_default: int = -1
    ner_infer_use_gpu: bool = False
    ner_safe_offload_classifier: bool = True

    seed: int = 42

cfg = Config()
set_seed(cfg.seed)
random.seed(cfg.seed)
np.random.seed(cfg.seed)
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    torch.cuda.empty_cache()
    gc.collect()

ROOT = Path(cfg.project_root)
PROCESSED_DIR = ROOT / cfg.processed_dir
TRAIN_PATH = PROCESSED_DIR / cfg.train_csv
VAL_PATH = PROCESSED_DIR / cfg.val_csv
TEST_PATH = PROCESSED_DIR / cfg.test_csv
BUILDER_SCRIPT = ROOT / cfg.builder_script

print(f"Target komputasi: {device}")
print(f"Project root: {ROOT}")
print(f"Processed dir: {PROCESSED_DIR}")



In [ ]:
# 5. Data Loading dari Processed Dataset (fallback build_dataset.py)

def assert_required_columns(df: pd.DataFrame, required: List[str], path: Path):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Kolom wajib tidak ditemukan di {path}: {missing}")


def load_processed_split(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"MISSING: {path}")
    df = pd.read_csv(path)
    assert_required_columns(df, ["text", "label", "source", "url_hash", "title_hash", "unit_type"], path)
    df = df.copy()
    df["text"] = df["text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df = df[df["text"] != ""].copy()
    df["label"] = pd.to_numeric(df["label"], errors="coerce").astype("Int64")
    df = df[df["label"].isin([0, 1])].copy()
    df["label"] = df["label"].astype(int)
    return df


if not (TRAIN_PATH.exists() and VAL_PATH.exists() and TEST_PATH.exists()):
    print("Processed split belum lengkap. Menjalankan builder script...")
    if not BUILDER_SCRIPT.exists():
        raise FileNotFoundError(f"MISSING: {BUILDER_SCRIPT}")

    cmd = [sys.executable, str(BUILDER_SCRIPT), "--root", str(ROOT), "--output-dir", str(PROCESSED_DIR)]
    print("Command:", " ".join(cmd))
    subprocess.run(cmd, check=True)


df_latih = load_processed_split(TRAIN_PATH)
df_validasi = load_processed_split(VAL_PATH)
df_uji = load_processed_split(TEST_PATH)

print("=== Ringkasan Processed Dataset ===")
for split_name, df_split in [("Latih", df_latih), ("Validasi", df_validasi), ("Uji", df_uji)]:
    print("")
    print(f"[{split_name}] rows={len(df_split)}")
    print(f"[{split_name}] distribusi label: {df_split['label'].value_counts().sort_index().to_dict()}")
    print(f"[{split_name}] source x label:")
    print(pd.crosstab(df_split["source"], df_split["label"]))

print("")
print("Contoh unit_type train:", df_latih["unit_type"].value_counts().to_dict())



In [6]:
# 6. Tokenisasi Dataset Hugging Face
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)


def tokenisasi_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=cfg.max_length)


ds_latih = Dataset.from_pandas(df_latih, preserve_index=False)
ds_validasi = Dataset.from_pandas(df_validasi, preserve_index=False)
ds_uji = Dataset.from_pandas(df_uji, preserve_index=False)

ds_latih = ds_latih.map(tokenisasi_batch, batched=True)
ds_validasi = ds_validasi.map(tokenisasi_batch, batched=True)
ds_uji = ds_uji.map(tokenisasi_batch, batched=True)

kolom_format = ["input_ids", "attention_mask", "label"]
ds_latih.set_format(type="torch", columns=kolom_format)
ds_validasi.set_format(type="torch", columns=kolom_format)
ds_uji.set_format(type="torch", columns=kolom_format)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if device == "cuda" else None,
)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/34438 [00:00<?, ? examples/s]

Map:   0%|          | 0/5538 [00:00<?, ? examples/s]

Map:   0%|          | 0/5538 [00:00<?, ? examples/s]

In [ ]:
# 7. Inisialisasi Model IndoBERT dan Pelatihan (Sinkron dengan Processed Dataset)
model = AutoModelForSequenceClassification.from_pretrained(
    cfg.model_name,
    num_labels=2,
    use_safetensors=False,
)
model.to(device)
model.config.use_cache = False
model.config.id2label = {0: "Fakta", 1: "Hoaks"}
model.config.label2id = {"Fakta": 0, "Hoaks": 1}


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    # Menyimpan dua key agar kompatibel lintas versi transformers
    return {
        "accuracy": float(acc),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_macro": float(f1_macro),
        "eval_f1_macro": float(f1_macro),
        "precision_weighted": float(p_weighted),
        "recall_weighted": float(r_weighted),
        "f1_weighted": float(f1_weighted),
        "cm_tn": int(tn),
        "cm_fp": int(fp),
        "cm_fn": int(fn),
        "cm_tp": int(tp),
    }


train_counts = df_latih["label"].value_counts().sort_index()
n_train = float(len(df_latih))
num_classes = 2
class_weights = []
for class_id in range(num_classes):
    count_class = float(train_counts.get(class_id, 1.0))
    class_weights.append(n_train / (num_classes * count_class))

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
if device == "cuda":
    class_weights_tensor = class_weights_tensor.to(device)

print(f"Class weights (0,1): {class_weights}")
print(f"Imbalance method: {cfg.imbalance_method}")


class ImbalanceTrainer(Trainer):
    def __init__(self, *args, class_weights=None, cfg_obj=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.cfg_obj = cfg_obj

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.cfg_obj.imbalance_method == "focal":
            ce_loss = F.cross_entropy(logits, labels, weight=self.class_weights, reduction="none")
            pt = torch.exp(-ce_loss)
            loss = ((1 - pt) ** self.cfg_obj.focal_gamma * ce_loss).mean()
        else:
            loss = F.cross_entropy(logits, labels, weight=self.class_weights)

        return (loss, outputs) if return_outputs else loss


argumen_kwargs = dict(
    output_dir=cfg.output_dir,
    per_device_train_batch_size=cfg.train_batch_size,
    per_device_eval_batch_size=cfg.eval_batch_size,
    gradient_accumulation_steps=cfg.grad_accumulation,
    learning_rate=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
    num_train_epochs=cfg.num_epochs,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=True if device == "cuda" else False,
    gradient_checkpointing=True,
    auto_find_batch_size=cfg.auto_find_batch_size,
    dataloader_num_workers=2,
    report_to="none",
)

training_arg_names = TrainingArguments.__init__.__code__.co_varnames
if "eval_strategy" in training_arg_names:
    argumen_kwargs["eval_strategy"] = "epoch"
else:
    argumen_kwargs["evaluation_strategy"] = "epoch"

argumen_latih = TrainingArguments(**argumen_kwargs)

trainer_kwargs = dict(
    model=model,
    args=argumen_latih,
    train_dataset=ds_latih,
    eval_dataset=ds_validasi,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor,
    cfg_obj=cfg,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=cfg.early_stopping_patience,
            early_stopping_threshold=cfg.early_stopping_threshold,
        )
    ],
)

trainer_arg_names = Trainer.__init__.__code__.co_varnames
if "processing_class" in trainer_arg_names:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = ImbalanceTrainer(**trainer_kwargs)

print("Memulai pelatihan (processed dataset + metrik lengkap)...")
trainer.train()

metric_validasi = trainer.evaluate(eval_dataset=ds_validasi)
metric_uji = trainer.evaluate(eval_dataset=ds_uji, metric_key_prefix="test")

print("")
print("=== Metrik Validasi ===")
for k, v in metric_validasi.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v}")

print("")
print("=== Metrik Uji ===")
for k, v in metric_uji.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v}")

# Logging kalibrasi ringan: margin logit pada set uji
pred_uji = trainer.predict(ds_uji)
logits_uji = pred_uji.predictions
label_pred = np.argmax(logits_uji, axis=-1)
margin = np.abs(logits_uji[:, 1] - logits_uji[:, 0])
calibration_info = {
    "logit_margin_mean": float(np.mean(margin)),
    "logit_margin_median": float(np.median(margin)),
    "logit_margin_p10": float(np.quantile(margin, 0.10)),
    "logit_margin_p90": float(np.quantile(margin, 0.90)),
    "pred_label_distribution": {
        "0": int((label_pred == 0).sum()),
        "1": int((label_pred == 1).sum()),
    },
}

os.makedirs(cfg.output_dir, exist_ok=True)
with open(os.path.join(cfg.output_dir, "calibration.json"), "w", encoding="utf-8") as f:
    json.dump(calibration_info, f, ensure_ascii=False, indent=2)

print("\n=== Calibration (Logit Margin) ===")
print(calibration_info)

trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print(f"Model final tersimpan di: {cfg.output_dir}")




In [ ]:
# 8. Inference Konsisten + Stress Test Produksi
print("Mempersiapkan modul Named Entity Recognition (NER) sekunder...")

ner_device = cfg.ner_infer_device_default
device_klasifikasi = device

if cfg.ner_infer_use_gpu and torch.cuda.is_available():
    ner_device = 0
    if cfg.ner_safe_offload_classifier:
        model.to("cpu")
        device_klasifikasi = "cpu"
        torch.cuda.empty_cache()
        gc.collect()
        print("Model klasifikasi dipindah ke CPU sebelum memuat NER di GPU (mode aman).")
else:
    ner_device = -1

if ner_device >= 0 and not torch.cuda.is_available():
    ner_device = -1

ner_pipeline = pipeline(
    "ner",
    model=cfg.ner_model_name,
    aggregation_strategy="simple",
    device=ner_device,
)

print(f"Device klasifikasi: {device_klasifikasi} | Device NER pipeline: {ner_device}")

PARAGRAPH_SPLIT_RE = re.compile(r"(?:\r?\n){2,}")
SENTENCE_SPLIT_RE = re.compile(r'[^.!?]+(?:[.!?]+(?:["\)\]]+)?)|[^.!?]+$')


def split_paragraphs(text: str) -> List[str]:
    paragraphs = [p.strip() for p in PARAGRAPH_SPLIT_RE.split(str(text).strip()) if p.strip()]
    if paragraphs:
        return paragraphs
    raw = str(text).strip()
    return [raw] if raw else []


def split_sentences(paragraph: str) -> List[str]:
    normalized = re.sub(r"\s+", " ", str(paragraph)).strip()
    if not normalized:
        return []
    sentences = [m.group(0).strip() for m in SENTENCE_SPLIT_RE.finditer(normalized)]
    return [s for s in sentences if s] or [normalized]


def classify_sentences(sentences: List[str]) -> List[Dict]:
    outputs = []
    model.eval()
    for s in sentences:
        inputs = tokenizer(
            s,
            return_tensors="pt",
            truncation=True,
            max_length=cfg.max_length,
            padding=True,
        ).to(device_klasifikasi)

        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).squeeze()
            pred = int(torch.argmax(probs).item())
            prob_hoax = float(probs[cfg.label_hoaks].item())
            prob_fakta = float(probs[cfg.label_fakta].item())
            conf = max(prob_hoax, prob_fakta)
            label = "Hoaks" if pred == cfg.label_hoaks else "Fakta"

        color = "orange" if conf < cfg.orange_threshold else ("red" if label == "Hoaks" else "green")
        outputs.append(
            {
                "text": s,
                "label": label,
                "prob_hoax": round(prob_hoax, 6),
                "prob_fakta": round(prob_fakta, 6),
                "confidence": round(conf, 6),
                "color": color,
            }
        )
    return outputs


def analyze_text_with_ner(text: str, include_ner: bool = True) -> Dict:
    paragraphs = split_paragraphs(text)
    analyzed_paragraphs = []
    all_sentences = []
    for p_idx, p in enumerate(paragraphs):
        sentences = split_sentences(p)
        cls = classify_sentences(sentences)
        analyzed_paragraphs.append({"paragraph_index": p_idx, "sentences": cls})
        all_sentences.extend(cls)

    ner_entities = []
    if include_ner:
        raw_ner = ner_pipeline(text)
        for ent in raw_ner:
            if float(ent.get("score", 0.0)) >= 0.60:
                ner_entities.append(
                    {
                        "word": ent.get("word", ""),
                        "entity_group": ent.get("entity_group", ""),
                        "score": round(float(ent.get("score", 0.0)), 6),
                    }
                )

    return {
        "paragraphs": analyzed_paragraphs,
        "summary": {
            "n_sentences": len(all_sentences),
            "n_hoaks": sum(1 for x in all_sentences if x["label"] == "Hoaks"),
            "n_fakta": sum(1 for x in all_sentences if x["label"] == "Fakta"),
        },
        "ner": ner_entities,
    }


print("\n=== STRESS TEST PRODUKSI ===")
if TEST_PATH.exists():
    df_stress = pd.read_csv(TEST_PATH)
    stress_rows = []
    for src in sorted(df_stress["source"].dropna().astype(str).unique()):
        subset = df_stress[df_stress["source"] == src]
        if subset.empty:
            continue
        stress_rows.append(subset.sample(n=min(cfg.stress_samples_per_source, len(subset)), random_state=cfg.seed))

    if stress_rows:
        stress_df = pd.concat(stress_rows, ignore_index=True)
        all_pred_labels = []
        for idx, row in stress_df.iterrows():
            text = str(row.get("text", ""))
            if not text.strip():
                continue
            result = analyze_text_with_ner(text, include_ner=False)
            sentence_preds = [s["label"] for p in result["paragraphs"] for s in p["sentences"]]
            all_pred_labels.extend(sentence_preds)
            print("")
            print(f"[Stress {idx+1}] source={row.get('source')} true_label={row.get('label')} pred_sentences={sentence_preds[:5]}")
            if result["paragraphs"] and result["paragraphs"][0]["sentences"]:
                first = result["paragraphs"][0]["sentences"][0]
                print(
                    f"  contoh: label={first['label']} prob_hoax={first['prob_hoax']} prob_fakta={first['prob_fakta']} conf={first['confidence']}"
                )

        if all_pred_labels:
            distribusi = {
                "Hoaks": int(sum(1 for x in all_pred_labels if x == "Hoaks")),
                "Fakta": int(sum(1 for x in all_pred_labels if x == "Fakta")),
            }
            print("\nDistribusi prediksi stress test:", distribusi)
            if distribusi["Hoaks"] == 0 or distribusi["Fakta"] == 0:
                print("WARNING: Prediksi stress test cenderung kolaps satu kelas.")
    else:
        print("Tidak ada data stress test yang bisa diambil dari processed test.")
else:
    print(f"MISSING: {TEST_PATH}")



In [ ]:
# 9. Uji Manual 6 Berita (Produksi)
berita_uji_6 = [
    """Gubernur DKI Jakarta Pramono Anung Wibowo disebut akan mengundang pihak-pihak terkait untuk membahas perizinan fasilitas padel di Jakarta. Langkah ini muncul setelah ada keluhan warga tentang kebisingan lapangan padel di dekat permukiman, dan pemerintah daerah disebut akan memetakan lokasi yang melanggar izin serta menyiapkan tindakan penertiban.""",
    """Beredar unggahan di media sosial yang mengklaim Kementerian Kehutanan membuka rekrutmen CPNS Polisi Kehutanan tahun 2026 secara besar-besaran. Unggahan itu menyebut kebutuhan personel sangat tinggi dan menyatakan lulusan SMA diprioritaskan, disertai ajakan agar publik segera mendaftar.""",
    """Sebuah pesawat pengangkut bahan bakar minyak dilaporkan jatuh di kawasan perbukitan Krayan, Kabupaten Nunukan, Kalimantan Utara. Laporan menyebut kepulan asap hitam terlihat setelah pesawat lepas landas usai mengantar pasokan BBM, dan saksi mata mengaku melihat pesawat sempat oleng sebelum jatuh pada kondisi cuaca yang berawan.""",
    """Beredar sebuah video yang memperlihatkan suasana bandara dengan aktivitas pesawat dan iring-iringan kendaraan mengantar tamu mancanegara. Video itu diklaim sebagai momen kedatangan rombongan Perserikatan Bangsa-Bangsa ke IKN Nusantara untuk agenda peresmian, dan narasi tersebut ramai dibagikan warganet.""",
    """PT Transjakarta dikabarkan melakukan modifikasi layanan pada empat rute mulai 21 Februari 2026, mencakup rute 4D, 6M, 9H, dan 11B. Perubahan ini disebut untuk mengoptimalkan layanan dan menjaga kenyamanan penumpang, dengan penyesuaian pada sebagian titik halte yang dilayani di rute tertentu.""",
    """Sebuah video di TikTok dipublikasikan dengan narasi “breaking” yang mengklaim terjadi kericuhan saat ribuan orang turun ke jalan untuk memprotes kebijakan Presiden AS Donald Trump. Unggahan tersebut memicu banyak komentar dan dibagikan ulang, dengan sebagian warganet menganggapnya sebagai rekaman aksi massa terbaru.""",
]

print("=== UJI MANUAL 6 BERITA ===")
for i, teks in enumerate(berita_uji_6, start=1):
    hasil = analyze_text_with_ner(teks, include_ner=True)
    semua_kalimat = [s for p in hasil["paragraphs"] for s in p["sentences"]]

    if semua_kalimat:
        avg_hoax = float(np.mean([x["prob_hoax"] for x in semua_kalimat]))
        avg_fakta = float(np.mean([x["prob_fakta"] for x in semua_kalimat]))
        label_doc = "Hoaks" if avg_hoax >= avg_fakta else "Fakta"
        confidence_doc = max(avg_hoax, avg_fakta)
    else:
        avg_hoax = 0.0
        avg_fakta = 0.0
        label_doc = "Fakta"
        confidence_doc = 0.0

    print("")
    print(f"[Berita {i}] Label dokumen: {label_doc} | conf={confidence_doc:.4f} | avg_hoax={avg_hoax:.4f} | avg_fakta={avg_fakta:.4f}")
    print(f"Ringkasan kalimat: {hasil['summary']}")

    if semua_kalimat:
        top = semua_kalimat[0]
        print(
            f"Kalimat contoh: label={top['label']} prob_hoax={top['prob_hoax']} prob_fakta={top['prob_fakta']} conf={top['confidence']}"
        )

    if hasil["ner"]:
        print("Entitas (maks 5):", hasil["ner"][:5])
    else:
        print("Entitas: tidak ada")



In [11]:
drive.mount('/content/drive')
folder_tujuan = '/content/drive/MyDrive/berita_hoax_output'
os.makedirs(folder_tujuan, exist_ok=True)

nama_file_zip = "indobert_hoax_ner_model_final"
nama_file_zip_lengkap = f"{nama_file_zip}.zip"

print(f"Sedang mengompresi direktori {cfg.output_dir}...")
shutil.make_archive(nama_file_zip, 'zip', cfg.output_dir)

jalur_sumber = nama_file_zip_lengkap
jalur_target = os.path.join(folder_tujuan, nama_file_zip_lengkap)

print(f"Mengunggah file {nama_file_zip_lengkap} ke Google Drive di folder 'berita_hoax_output'...")
try:
    shutil.copy(jalur_sumber, jalur_target)
    print(f"Proses unggah berhasil. File tersimpan di: {jalur_target}")
except Exception as e:
    print(f"Terjadi kesalahan saat mengunggah ke Google Drive: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sedang mengompresi direktori indobert_hoax_ner_model...
Mengunggah file indobert_hoax_ner_model_final.zip ke Google Drive di folder 'berita_hoax_output'...
Proses unggah berhasil. File tersimpan di: /content/drive/MyDrive/berita_hoax_output/indobert_hoax_ner_model_final.zip


In [12]:
!pip install -q huggingface_hub

from huggingface_hub import login, HfApi
from google.colab import userdata

try:
    token_kredensial = userdata.get('HF_TOKEN')
    login(token=token_kredensial)
except Exception:
    print("Token HF_TOKEN tidak ditemukan pada Colab Secrets.")

api_huggingface = HfApi()
id_repositori_target = "fjrmhri/Deteksi_Hoax_TA"
direktori_sumber_artefak = cfg.output_dir

api_huggingface.create_repo(repo_id=id_repositori_target, private=False, exist_ok=True)

print(f"Memulai unggahan dari {direktori_sumber_artefak} menuju {id_repositori_target}...")
api_huggingface.upload_folder(
    folder_path=direktori_sumber_artefak,
    repo_id=id_repositori_target,
    repo_type="model",
    commit_message="Pembaruan model IndoBERT deteksi hoaks"
)
print(f"Transmisi selesai. URL Repositori: https://huggingface.co/{id_repositori_target}")

Memulai unggahan dari indobert_hoax_ner_model menuju fjrmhri/Deteksi_Hoax_TA...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...kpoint-3231/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...kpoint-1077/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

...

  ...r_model/model.safetensors:   0%|          | 92.7kB /  442MB            

  ...nt-3231/model.safetensors:   0%|          |  560kB /  442MB            

  ...ckpoint-1077/scheduler.pt:   6%|5         |  86.0B / 1.47kB            

  ...nt-1077/training_args.bin:   6%|5         |   306B / 5.20kB            

Transmisi selesai. URL Repositori: https://huggingface.co/fjrmhri/Deteksi_Hoax_TA
